# PA3 Task 2 — Kaggle runner (SigLIP → connector → Qwen3-0.6B VLM, visual grounding)

One Kaggle version (*Save Version → Save & Run All (Commit)*, ≤ 12 h) runs the whole task:

| Phase | GPU 0 | GPU 1 |
|---|---|---|
| 1 | data selection + image download → SigLIP feature cache → preflight | (idle) |
| 2 | train `main` (196 visual tokens) → final evaluation | train `compress49` (optional 49-token run) → final evaluation |
| 3 | collect logs → tables + figures → zip the small outputs | |

Training stops cleanly before `MAX_HOURS` (resumable state saved). If anything is unfinished, run another version with
`PREV_OUTPUT` = the previous version's output: finished pieces are skipped, paused runs resume exactly.

Notebook settings: **Accelerator = GPU T4 ×2**, **Internet = On**.

In [ ]:
# ---------------- settings (edit these) ----------------
REPO_URL       = "https://github.com/wasaymj/ATML.git"
BRANCH         = "main"
PREV_OUTPUT    = ""        # e.g. "/kaggle/input/pa3-task2-runner" (Add Input -> Your Work -> this notebook) to resume
MAX_HOURS      = 9.5       # per training process; leaves time for final evaluation + report inside Kaggle's 12 h
RUN_PREFLIGHT  = True      # ~5-10 min of checks before training (recommended on the first run)
RUN_COMPRESS49 = True      # optional 196 -> 49 visual-token experiment on the second GPU

import os, subprocess, sys, time, shutil, glob
WORK = "/kaggle/working"
REPO = f"{WORK}/ATML"
TASK = f"{REPO}/PA3/Task 2"
CKPT = f"{WORK}/ckpts"
LOGS = f"{WORK}/logs"
os.makedirs(LOGS, exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
print(shutil.disk_usage(WORK))

In [ ]:
# ---------------- get the code ----------------
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# ---------------- install pinned dependencies (torch stays Kaggle's CUDA build) ----------------
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{TASK}/requirements.txt"])
print("pip returncode", r.returncode)   # Kaggle may print resolver warnings about unrelated preinstalled packages
print(subprocess.run([sys.executable, "-c",
    "import torch, transformers, peft, PIL; print('torch', torch.__version__, '| transformers', transformers.__version__, "
    "'| peft', peft.__version__, '| pillow', PIL.__version__)"], capture_output=True, text=True).stdout)

In [ ]:
# ---------------- restore the previous version's outputs (resume) ----------------
def copy_tree(src, dst):
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("restored", src, "->", dst)
if PREV_OUTPUT:
    copy_tree(f"{PREV_OUTPUT}/ckpts", CKPT)
    for sub in ("data/ids", "data/build", "data/images", "features", "results", "label_audit"):
        copy_tree(f"{PREV_OUTPUT}/ATML/PA3/Task 2/{sub}", f"{TASK}/{sub}")
print(shutil.disk_usage(WORK))

In [ ]:
# ---------------- helpers ----------------
ENV = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false")

def run(*args, gpu=None):
    # Run one step in the foreground; its log is streamed into the notebook.
    env = dict(ENV, **({"CUDA_VISIBLE_DEVICES": str(gpu)} if gpu is not None else {}))
    cmd = [sys.executable, "scripts/run_task2.py", *args, "--ckpt_root", CKPT]
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, cwd=TASK, env=env).returncode

def queue(name, gpu, steps):
    # Run a list of steps one after another in the background on one GPU.
    lines = [f'cd "{TASK}"']
    for args in steps:
        lines.append(f'{sys.executable} scripts/run_task2.py {args} --ckpt_root {CKPT} '
                     f'|| echo "STEP EXITED NON-ZERO: {args}"')
    script = f"{LOGS}/{name}.sh"
    open(script, "w").write("\n".join(lines) + "\n")
    log = open(f"{LOGS}/{name}.log", "a")
    print(f"[{name}] GPU {gpu}: {steps}\n   log: {LOGS}/{name}.log", flush=True)
    return subprocess.Popen(["bash", script], stdout=log, stderr=subprocess.STDOUT, env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu)))

def wait(procs, every_min=10):
    while any(p.poll() is None for p in procs.values()):
        time.sleep(every_min * 60)
        for name, p in procs.items():
            tail = open(f"{LOGS}/{name}.log", errors="replace").read().splitlines()[-2:]
            print(f"--- {name} ({'running' if p.poll() is None else 'finished'})", *tail, sep="\n   ", flush=True)
        print(shutil.disk_usage(WORK), flush=True)
    print({n: p.returncode for n, p in procs.items()})
    # A failed step does not stop its queue (the next step still runs), so report failures explicitly.
    for name in procs:
        lines = open(f"{LOGS}/{name}.log", errors="replace").read().splitlines()
        failed = [l for l in lines if l.startswith("STEP EXITED NON-ZERO")]
        errors = [l for l in lines if "Error:" in l and not l.startswith(" ")]
        if failed:
            print(f"!!! {name}: {len(failed)} step(s) failed", *failed, *errors[-5:], sep="\n   ", flush=True)
        else:
            print(f"OK  {name}: every step exited normally", flush=True)

## Phase 1: data, features, preflight
`data` downloads ~300 MB of annotations, selects every split (seed 6304), saves ids/stats, downloads only the ~9.3k
selected COCO images and writes the blind label-audit sheets. `features` caches SigLIP features (FP16) and releases
the encoder. Preflight checks split isolation, the schedule, chat template + visual insertion, no-image rebuild,
label masks, calibration, gradient flow through the frozen backbone, frozen sets, memory, and the evaluation paths.

In [ ]:
assert run("--step", "data", gpu=0) == 0
assert run("--step", "features", gpu=0) == 0
if RUN_PREFLIGHT:
    assert run("--step", "preflight", gpu=0) == 0, "Preflight failed - read the log above before training."
# the raw annotation json (~1.2 GB) is no longer needed once the splits are selected (ids + build files are kept)
shutil.rmtree(f"{TASK}/data/raw", ignore_errors=True)
print(shutil.disk_usage(WORK))

## Phase 2: training (both GPUs) and final evaluation

In [ ]:
mh = f"--max_hours {MAX_HOURS}"
procs = {"gpu0_main": queue("gpu0_main", 0, [f"--step train --run main {mh}", "--step final_eval --run main"])}
if RUN_COMPRESS49:
    procs["gpu1_compress49"] = queue("gpu1_compress49", 1, [f"--step train --run compress49 {mh}",
                                                            "--step final_eval --run compress49"])
wait(procs)

## Phase 3: status, report, package

In [ ]:
run("--step", "status")
run("--step", "report")       # also copies the small training logs into results/training/
run("--step", "status")

In [ ]:
# ---------------- package the small, git-tracked outputs for download ----------------
shutil.make_archive(f"{WORK}/task2_results", "zip", TASK, "results")
shutil.make_archive(f"{WORK}/task2_data_ids", "zip", TASK, "data/ids")
if os.path.exists(f"{TASK}/label_audit"):
    shutil.make_archive(f"{WORK}/task2_label_audit", "zip", TASK, "label_audit")
print(sorted(os.listdir(WORK)))

In [ ]:
# Inline preview of the tables and figures
from IPython.display import Markdown, Image, display
for t in sorted(glob.glob(f"{TASK}/results/tables/*.md")):
    display(Markdown(f"### {os.path.basename(t)}\n" + open(t, encoding="utf-8").read()))
for fig in sorted(glob.glob(f"{TASK}/results/figures/*.png")):
    display(Image(fig, width=800))